In [ ]:
import pandas as pd
import numpy as np
import ast

In [ ]:
movies = pd.read_csv('movies_metadata.csv')
keywords=pd.read_csv('keywords.csv')
credits = pd.read_csv('credits.csv',low_memory=False)

/tmp/ipykernel_4835/2466890255.py:1: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  movies = pd.read_csv('movies_metadata.csv')


In [ ]:
movies.columns

Index(['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'video',
       'vote_average', 'vote_count'],
      dtype='object')

In [ ]:
movies = movies[
    [
        'id',
        'title',
        'overview',
        'genres',
        'poster_path',
        'vote_average',
        'vote_count',
        'release_date',
        'runtime'
    ]
]

**Selecting top 25000 movies**

In [ ]:
movies = movies.nlargest(
    25000,
    "vote_count"
).reset_index(drop=True)

print(movies.shape)

(25000, 9)


In [ ]:
movies['id'] = movies['id'].astype(str)
credits['id'] = credits['id'].astype(str)
keywords['id'] = keywords['id'].astype(str)

In [ ]:
movies = movies.merge(keywords, on='id')
movies = movies.merge(credits, on='id')

Converting string into list, to serialize later

In [ ]:
def convert(obj):

    L = []

    for i in ast.literal_eval(obj):
        L.append(i['name'])

    return L

In [ ]:
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

**Extracting Top 3 cast members**

In [ ]:
def convert_cast(obj):

    L = []

    counter = 0

    for i in ast.literal_eval(obj):

        if counter != 3:
            L.append(i['name'])
            counter += 1

        else:
            break

    return L

In [ ]:
movies['cast'] = movies['cast'].apply(convert_cast)

**Extracting Director**

In [ ]:
def fetch_director(obj):

    L = []

    for i in ast.literal_eval(obj):

        if i['job'] == 'Director':
            L.append(i['name'])
            break

    return L

In [ ]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [ ]:
movies['overview'] = movies['overview'].fillna('')

In [ ]:
movies['overview'] = movies['overview'].apply(lambda x: x.split())

In [ ]:
movies['genres'] = movies['genres'].apply(
    lambda x: [i.replace(" ", "") for i in x]
)

movies['keywords'] = movies['keywords'].apply(
    lambda x: [i.replace(" ", "") for i in x]
)

movies['cast'] = movies['cast'].apply(
    lambda x: [i.replace(" ", "") for i in x]
)

movies['crew'] = movies['crew'].apply(
    lambda x: [i.replace(" ", "") for i in x]
)

Generating the tag . ***Overview, genre, keywords, top 3 cast, and director***

In [ ]:
movies['tag'] = (
    movies['overview']
    + movies['genres']
    + movies['keywords']
    + movies['cast']
    + movies['crew']
)

In [ ]:
new_data = movies[
    [
        'id',
        'title',
        'tag',
        'overview',
        'poster_path',
        'vote_average',
        'release_date'
    ]
].copy()

In [ ]:
new_data['tag'] = new_data['tag'].apply(
    lambda x: " ".join(x)
)

In [ ]:
new_data['tag'] = new_data['tag'].fillna('')
new_data['title'] = new_data['title'].fillna('')

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
import pickle

In [ ]:
tfidf = TfidfVectorizer(
    max_features=3000,
    stop_words='english'
)

vectors = tfidf.fit_transform(new_data['tag'])

In [ ]:
model = NearestNeighbors(
    metric='cosine',
    algorithm='brute',
    n_neighbors=11
)

model.fit(vectors)

NearestNeighbors(algorithm='brute', metric='cosine', n_neighbors=11)

In [ ]:
titles = new_data[['title']]

In [ ]:
def recommend(movie):

    movie = movie.lower()

    matches = titles[
      titles['title']
      .fillna('')
      .str.lower()
      .str.contains(movie)
  ]

    if matches.empty:
        return []

    movie_index = matches.index[0]

    distances, indices = model.kneighbors(
        vectors[movie_index],
        n_neighbors=11
    )

    recommended_movies = []

    for i in indices[0][1:]:

        movie_data = movies.iloc[i]

        recommended_movies.append({
            "id": int(movie_data.id),

            "title": movie_data.title,

            "overview": (
                " ".join(movie_data.overview)
                if isinstance(movie_data.overview, list)
                else movie_data.overview
            ),

            "poster": (
                "https://image.tmdb.org/t/p/w500"
                + str(movie_data.poster_path)
                if pd.notna(movie_data.poster_path)
                else None
            ),

            "rating": (
                float(movie_data.vote_average)
                if pd.notna(movie_data.vote_average)
                else None
            ),

            "release_date": movie_data.release_date,

            "genres": (
                ", ".join(movie_data.genres)
                if isinstance(movie_data.genres, list)
                else movie_data.genres
            )
        })

    return recommended_movies

In [ ]:
recommend("Interstellar")

[{'id': 29570,
  'title': 'Journey to the Seventh Planet',
  'overview': 'A space expedition to Uranus is menaced by a giant brain that can make illusions come true.',
  'poster': 'https://image.tmdb.org/t/p/w500/teVKcwWifphk2jDzFEuVMJlBiLz.jpg',
  'rating': 4.4,
  'release_date': '1962-03-10',
  'genres': 'Adventure, Fantasy, Horror, ScienceFiction'},
 {'id': 188507,
  'title': 'Space Warriors',
  'overview': 'The son of a retired astronaut competes to win a seat on the next space shuttle.',
  'poster': 'https://image.tmdb.org/t/p/w500/56mvfh550R8H8VMdSHNK3flPo6u.jpg',
  'rating': 4.6,
  'release_date': '2013-04-26',
  'genres': 'ScienceFiction, Adventure, Family'},
 {'id': 13766,
  'title': 'SpaceCamp',
  'overview': 'Andie Bergstrom, an astronaut eagerly awaiting her first trip to space, runs a summer camp for teenagers with her NASA-employed husband, Zach. One night during an engine test, Andie and four teenage campers are accidentally shot into space. Together, the group -- which 

In [ ]:
import pickle

pickle.dump(
    movies,
    open('movies.pkl', 'wb')
)

pickle.dump(
    titles,
    open(
        'titles.pkl',
        'wb'
    )
)
pickle.dump(
    model,
    open('model.pkl', 'wb')
)
pickle.dump(
    tfidf,
    open('tfidf.pkl', 'wb')
)
vectors = vectors.astype("float32")
pickle.dump(
    vectors,
    open('vectors.pkl', 'wb')
)

*Tests....*

In [ ]:
moviesfull = pd.read_csv('movies_metadata.csv')

/tmp/ipykernel_4835/2244205811.py:1: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  moviesfull = pd.read_csv('movies_metadata.csv')


In [ ]:
movies.columns

Index(['id', 'title', 'overview', 'genres', 'poster_path', 'vote_average',
       'vote_count', 'release_date', 'runtime', 'keywords', 'cast', 'crew',
       'tag'],
      dtype='object')

In [ ]:
movies['runtime'] = moviesfull['runtime']

In [ ]:
pickle.dump(
    movies,
    open('movies.pkl', 'wb')
)

In [ ]:
import os

for f in [
    "movies.pkl",
    "titles.pkl",
    "vectors.pkl",
    "model.pkl"
]:
    print(
        f,
        round(os.path.getsize(f)/(1024*1024),2),
        "MB"
    )

movies.pkl 24.5 MB
titles.pkl 0.45 MB
vectors.pkl 4.38 MB
model.pkl 6.52 MB


In [ ]:
import os

for f in [
    "movies.pkl",
    "titles.pkl",
]:
    print(
        f,
        round(os.path.getsize(f)/(1024*1024),2),
        "MB"
    )

movies.pkl 24.5 MB
titles.pkl 0.45 MB
